In [1]:
import pandas as pd

In [2]:
raw_path ="insurance_deidentified.csv"
df=pd.read_csv(raw_path)

In [3]:
print (df.columns)

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='object')


In [4]:
#cleaning
df.columns=df.columns.str.strip().str.capitalize().str.replace(" ","_")


In [5]:
df=df.drop_duplicates()


In [6]:
df=df.fillna("UNKNOWN")

In [7]:
df.isnull().sum()

Age         0
Sex         0
Bmi         0
Children    0
Smoker      0
Region      0
Charges     0
dtype: int64

In [8]:
df['Smoker'] = df['Smoker'].str.capitalize()
df['Sex'] = df['Sex'].str.capitalize()
df['Region'] = df['Region'].str.capitalize()


In [9]:
curated_path="cleaned_insurance.csv"
df.to_csv(curated_path, index= False)

In [10]:
print(" Insurance date curated successfully")

 Insurance date curated successfully


In [11]:
import boto3
import pandas as pd
from io import StringIO



In [12]:
bucket_name= "my-healthcare-bucket-bishal-2026"
raw_key="rawfile/insurance/insurance_deidentified.csv"
curated_key="curated/insurance/cleaned_insurance.csv"


In [13]:
#initialize S3 client
s3= boto3.client('s3')

In [14]:
#Function to read CSV from s3
def read_csv_from_s3(bucket,key):
    obj=s3.get_object(Bucket=bucket_name, Key=key)
    df= pd.read_csv(obj['Body'])
    return df


In [15]:
#Read raw Csv
raw_df=read_csv_from_s3(bucket_name, raw_key)
print("Raw Data Preview:")
print(raw_df.head())


Raw Data Preview:
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [16]:
#read curate csv
curated_df=read_csv_from_s3(bucket_name, curated_key)
print("\nCurated Data Preview:")
print(curated_df.head())


Curated Data Preview:
   Age     Sex     Bmi  Children Smoker     Region      Charges
0   19  Female  27.900         0    Yes  Southwest  16884.92400
1   18    Male  33.770         1     No  Southeast   1725.55230
2   28    Male  33.000         3     No  Southeast   4449.46200
3   33    Male  22.705         0     No  Northwest  21984.47061
4   32    Male  28.880         0     No  Northwest   3866.85520


In [17]:
#cleaning data
curated_df = raw_df.dropna()
#remove duplicate
curated_df = curated_df.drop_duplicates()

In [18]:
#standardize columns name
curated_df.columns=curated_df.columns.str.capitalize()
print("Curated rows:", len(curated_df))

Curated rows: 1337


In [19]:
#Curate data back to S3
csv_buffer=StringIO()
curated_df.to_csv(csv_buffer, index=False)


In [20]:
s3.put_object(
    Bucket=bucket_name,
    Key=curated_key,
    Body=csv_buffer.getvalue()
    )

{'ResponseMetadata': {'RequestId': 'TG7MX7H6VQ70CWDK',
  'HostId': 'pdttqsCnRgF2Z3Iu9AGFoS++htY1+CWO2j1u7chj/OA4a1jRlaPZ0qk2P2s6SHQuFDG2Jn7q35bqHlsmuzo/pWGZHGu/irBR',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'pdttqsCnRgF2Z3Iu9AGFoS++htY1+CWO2j1u7chj/OA4a1jRlaPZ0qk2P2s6SHQuFDG2Jn7q35bqHlsmuzo/pWGZHGu/irBR',
   'x-amz-request-id': 'TG7MX7H6VQ70CWDK',
   'date': 'Tue, 10 Feb 2026 13:14:42 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"9ffb054dd6f5696e2bbabc502693741f"',
   'x-amz-checksum-crc32': 'KBrtnQ==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"9ffb054dd6f5696e2bbabc502693741f"',
 'ChecksumCRC32': 'KBrtnQ==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [21]:
print("Curated Data Upload Successfully!!")

Curated Data Upload Successfully!!


In [22]:
#Validate curated data.......
validated_df=read_csv_from_s3(bucket_name,curated_key)
print("Validated Curated Data:")
print(validated_df.head())

Validated Curated Data:
   Age     Sex     Bmi  Children Smoker     Region      Charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520
